# 02 — PGD Adversarial Attack Evaluation

Runs projected gradient descent (PGD, L∞) on a fixed MNIST evaluation subset
and records per-sample clean/adversarial predictions in a CSV.

**Thesis context — WP1:** Measures empirical robustness of the trained model.
The fixed 100-sample evaluation split (`assets/splits/mnist_eval_100.json`)
ensures reproducibility across experiments.

**Prerequisites:** Either run `01_train_mnist.ipynb` first, or mount Google Drive
where a checkpoint already exists and set `CKPT_PATH` below.

In [ ]:
!pip install -q torch torchvision numpy pandas pyyaml tqdm ortools

## 1 — Library code

In [ ]:
from __future__ import annotations

import json
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


# ── Reproducibility ────────────────────────────────────────────────────────────
def set_seed(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)


# ── Data ───────────────────────────────────────────────────────────────────────
def get_mnist_datasets(data_dir: str | Path = "data"):
    data_dir = Path(data_dir)
    tfm = transforms.ToTensor()
    train_ds = datasets.MNIST(root=str(data_dir), train=True,  download=True, transform=tfm)
    test_ds  = datasets.MNIST(root=str(data_dir), train=False, download=True, transform=tfm)
    return train_ds, test_ds


# ── Evaluation split ───────────────────────────────────────────────────────────
@dataclass(frozen=True)
class Split:
    seed: int
    indices: list[int]


def load_split(path: str | Path) -> Split:
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    return Split(seed=int(obj["seed"]), indices=[int(i) for i in obj["indices"]])


def save_split(path: str | Path, split: Split) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(
        json.dumps({"seed": split.seed, "indices": split.indices}, indent=2) + "\n",
        encoding="utf-8",
    )


def ensure_mnist_eval_split(path: str | Path, seed: int = 1234, n: int = 100) -> Split:
    p = Path(path)
    if p.exists():
        return load_split(p)
    indices = random.Random(seed).sample(range(10_000), n)
    split = Split(seed=seed, indices=indices)
    save_split(p, split)
    return split


# ── Models ─────────────────────────────────────────────────────────────────────
class MnistMlp(nn.Module):
    def __init__(self, in_dim=784, h1=128, h2=64, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(int(in_dim), int(h1))
        self.fc2 = nn.Linear(int(h1), int(h2))
        self.fc3 = nn.Linear(int(h2), int(num_classes))
        self.relu = nn.ReLU()

    def forward(self, x):
        if x.ndim == 4:
            x = x.view(x.shape[0], -1)
        return self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))

    def linear_layers(self):
        return [self.fc1, self.fc2, self.fc3]


class CnnSmall(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(32*7*7, 64), nn.ReLU(), nn.Linear(64, int(num_classes)),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ── Checkpoint I/O ─────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class CheckpointMeta:
    model_type: str
    model_kwargs: dict
    run_name: str


def build_model(model_type, model_kwargs):
    if model_type == "mlp":       return MnistMlp(**model_kwargs)
    if model_type == "cnn_small": return CnnSmall(**model_kwargs)
    raise ValueError(f"Unknown model_type={model_type!r}")


def load_checkpoint(path, map_location="cpu"):
    payload = torch.load(str(path), map_location=map_location)
    raw = payload["meta"]
    meta = CheckpointMeta(
        model_type=str(raw["model_type"]),
        model_kwargs=dict(raw.get("model_kwargs", {})),
        run_name=str(raw.get("run_name", "run")),
    )
    model = build_model(meta.model_type, meta.model_kwargs)
    model.load_state_dict(payload["model_state_dict"])
    return model, meta, payload.get("metrics", {})


# ── PGD attack ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def _clamp_linf(x, x0, eps):
    lo = (x0 - float(eps)).clamp(0.0, 1.0)
    hi = (x0 + float(eps)).clamp(0.0, 1.0)
    return x.clamp(lo, hi)


def pgd_linf(model, x0, y, eps, steps, step_size, random_start=False):
    model.eval()
    x0 = x0.detach()
    x  = x0.clone()
    if random_start:
        x = _clamp_linf(x + (2 * torch.rand_like(x) - 1) * float(eps), x0, eps)
    for _ in range(int(steps)):
        x.requires_grad_(True)
        loss = F.cross_entropy(model(x), y, reduction="sum")
        grad = torch.autograd.grad(loss, x)[0]
        with torch.no_grad():
            x = _clamp_linf(x + float(step_size) * grad.sign(), x0, eps)
        x = x.detach()
    return x


print("Library code loaded")

## 2 — Configuration

Set `CKPT_PATH` to point to a trained model checkpoint.
If running after `01_train_mnist.ipynb` in the same session, the default should work.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
CKPT_PATH    = "runs/mlp_mnist/model.pt"          # path to trained checkpoint
SUBSET_PATH  = "assets/splits/mnist_eval_100.json" # fixed evaluation split
DATA_DIR     = "data"
EPS          = 0.03     # L-inf perturbation radius
STEPS        = 40       # PGD iterations
STEP_SIZE    = 0.01     # step size (alpha)
RANDOM_START = True     # random initialisation
BATCH_SIZE   = 64
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
# ──────────────────────────────────────────────────────────────────────────────

print(f"Device: {DEVICE} | eps={EPS} | steps={STEPS}")

## 3 — Ensure evaluation split exists

In [ ]:
split = ensure_mnist_eval_split(SUBSET_PATH)
print(f"Evaluation split: {len(split.indices)} samples (seed={split.seed})")

## 4 — Load model

In [ ]:
model, meta, _ = load_checkpoint(CKPT_PATH, map_location=DEVICE)
model.to(DEVICE).eval()
print(f"Loaded model: {meta.model_type!r} (run={meta.run_name!r})")

## 5 — Run PGD evaluation

In [ ]:
@torch.no_grad()
def batch_metrics(model, x, y):
    logits = model(x)
    loss   = F.cross_entropy(logits, y, reduction="none")
    return logits.argmax(dim=1), loss


_, test_ds = get_mnist_datasets(DATA_DIR)
sub_ds     = Subset(test_ds, split.indices)
loader     = DataLoader(sub_ds, batch_size=int(BATCH_SIZE), shuffle=False, num_workers=0)

rows  = []
start = time.time()

for batch_idx, (x0, y) in enumerate(loader):
    x0, y = x0.to(DEVICE), y.to(DEVICE)
    clean_pred, clean_loss = batch_metrics(model, x0, y)
    x_adv = pgd_linf(
        model, x0=x0, y=y, eps=float(EPS),
        steps=int(STEPS), step_size=float(STEP_SIZE), random_start=bool(RANDOM_START),
    )
    adv_pred, adv_loss = batch_metrics(model, x_adv, y)

    base = batch_idx * int(BATCH_SIZE)
    for i in range(y.shape[0]):
        rows.append({
            "index":        int(split.indices[base + i]),
            "y":            int(y[i].item()),
            "clean_pred":   int(clean_pred[i].item()),
            "adv_pred":     int(adv_pred[i].item()),
            "success":      int(adv_pred[i].item()) != int(y[i].item()),
            "clean_loss":   float(clean_loss[i].item()),
            "adv_loss":     float(adv_loss[i].item()),
            "eps":          float(EPS),
            "steps":        int(STEPS),
            "step_size":    float(STEP_SIZE),
            "random_start": bool(RANDOM_START),
            "run_name":     meta.run_name,
        })

elapsed = time.time() - start
df = pd.DataFrame(rows)

Path("results").mkdir(parents=True, exist_ok=True)
out_path = f"results/pgd_{meta.run_name}_eps{EPS:.4f}.csv"
df.to_csv(out_path, index=False)

asr = df["success"].mean()
print(f"Attack success rate (ASR): {asr:.4f} ({df['success'].sum()}/{len(df)})")
print(f"Clean accuracy on subset: {(df['clean_pred'] == df['y']).mean():.4f}")
print(f"Results saved to: {out_path} ({len(df)} rows) in {elapsed:.2f}s")

## 6 — Summary

In [ ]:
print(df.groupby("y")[["success", "clean_loss", "adv_loss"]].mean().round(4).to_string())
df.head(10)